In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import allmusic
mio   = allmusic.MusicDBIO(verbose=False)
webio = allmusic.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant AllMusic DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/AllMusic


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
knownArtists       = mio.data.getSummaryNameData()
searchArtists      = mio.data.getSearchArtistData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Local Artists:             {0}".format(len(localArtists.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

AllMusic Search Results
   Global Master Search:      761013
   Global Master Search Data: 0
   Local Artists:             477570
   Errors:                    521
   Search Artists:            1803604
   Known Summary IDs:         718650


# Search For New Artists

In [ ]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [ ]:
useKnownData  = False
useMasterData = True

if useKnownData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].notna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import PanDBIO
    pdbio = PanDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]
    del pdbio

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  50696
#   Artist Names To Get:  42803
#   Artist Names To Get:  33587
#   Artist Names To Get:  21888
#   Artist Names To Get:  10630

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "7:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue
    if searchedForErrors.get(artistName) is not None:
        continue

    try:
        response = webio.getArtistSearchData(artistName=artistName)
    except:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(60)
        continue
        
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        webio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
masterArtists.save(data=masterArtistsDict)
masterArtistsData.save(data=masterArtistsDataDict)

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        searchData = {}
        for searchTerm,searchTermData in mad.items():
            if isinstance(searchTermData,list):
                for item in searchTermData:
                    if isinstance(item,dict):
                        artistID = item['id'][2:] if isinstance(item.get('id'),str) else None
                        if artistID is not None:
                            searchData[artistID] = {k: v for k,v in item.items() if k in ['name','ref']}
        df = DataFrame(searchData).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [ ]:
mad = masterArtistsData.get()
df = getResponseDataFrame(mad)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = allmusic.MusicDBIO(local=False).data.getSearchArtistData()
    prevNewArtists = len(searchDF[~searchDF.index.isin(knownArtists.index)])
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("  ==> {0} Old Artists".format(searchDF[searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} New Artists".format(searchDF[~searchDF.index.isin(knownArtists.index)].shape[0]))
    print("  ==> {0} Delta New Artists".format(len(searchDF[~searchDF.index.isin(knownArtists.index)])-prevNewArtists))
    print("Saving Data")
    allmusic.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)
    
#Found 1628828 Unique Results
#Found 1639245 Unique Results
#Found 1664572 Unique Results
#Found 1674908 Unique Results

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [6]:
mio   = allmusic.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = allmusic.RawWebData(debug=False)

In [63]:
useKnown = False

artistNames       = searchArtists
localArtistsDict  = localArtists.get()
artistNamesToGet  = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
if useKnown is True:
    pdbio = PanDBIO()
    pdbio.setData()
    matchedIDs = pdbio.getNotNaDBIDs(db)[db]
    artistNamesToGet = artistNamesToGet[artistNamesToGet.index.isin(matchedIDs)]
    del pdbio

print("# {0} Search Results".format(db))
print("#   Available Names:      {0}".format(len(artistNames)))
print("#   Known Artist Names:   {0}".format(len(localArtistsDict)))
print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))

#   Artist Names To Get:  1013559

# AllMusic Search Results
#   Available Names:      1803604
#   Known Artist Names:   800020
#   Artist Names To Get:  1003584


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["name"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting AllMusic ArtistIDs] Start    ==> Time Is 2022-05-24 07:14:11
========================= termTime(day=today,time=9:00pm) =========================
   ====> Terminate Time Set To 2022-05-24 21:00:00 <====
   ====> Will Terminate Process 13 Hours and 45 Minutes From Now
Getting Ryan Orr (0001428855)                             True
Getting Pelle B (0002601072)                              True
Getting Raymond Hansen (0001467007)                       True
Getting Young Ran (0000688353)                            True
Getting Young Ron (0001508788)                            True
Getting Young Ronnie (0003311487)                         True
Getting Ron Young (0000833556)                            True
Getting Ron Young (0001492675)                            True
Getting Ron Young (0001928153)                            True
Getting Ron Young (0003188472)                            True
Getting Ronnie Young (0003694421)                         True
Getting Lisa Renée Youn

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Feel Feat (0001625898)                            True
Getting Alice Feat (0001894360)                           True
Getting Karen Feat (0002364176)                           True
Getting Mick Feat (0002436292)                            True
Getting The Bare Feat (0003384399)                        True
Getting Michael Feat (0003480319)                         True
Getting Jan Feat. (0003572523)                            True
Getting Pixie feat. Rhallia (0000293356)                  True
Getting Season feat. Ernesto (0000315688)                 True
Getting Charles Fink (0003482400)                         True
Getting The Charles E. Funk Rebellion (0000068497)        True
Getting Angel Lores Jiménez (0002376556)                  True
Getting Angel Lares Jiménez (0003378809)                  True
Getting Angel Llar (0003529320)                           True
Getting Miguel Ángel Leris Claramunt (0003385300)         True
Getting Laurie Angelo (0000421991)                     

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Bas Dubbelman (0003919238)                        True
Getting The Nolan Bishop Family Band (0003478331)         True
Getting Bishop Baby Ray & the Loose Gravel Band (0000429271)True
Getting Hans Herbert Fiedler (0001644850)                 True
Getting Hans Herbert Gebhard (0001203537)                 True
Getting Hans Herbert Emmerich (0002415774)                True
Getting Hans Herbert (0003428553)                         True
Getting Tom Jacobs (0000509125)                           True
Getting Tom Jacob (0003160788)                            True
Getting Tom Jacobus (0003324005)                          True
Getting Tom Jakab (0003941812)                            True
Getting Buzz Boyzz (0000630419)                           True
Getting Hometown Boyzz (0000267846)                       True
Getting Karibe Boyzz (0001805259)                         True
Getting 334 Boyzz (0000442409)                            True
Getting Donald Paul Fountain (0003222860)            

...

Getting Corona Barathri (0004035828)                      True
Getting Corona Lantern (0004077072)                       True
Getting Mario Corona (0001651680)                         True
Getting Claudio Corona (0003739564)                       True
Getting Mary Virginia Taylor (0003908845)                 True
Getting Mary Lee Taylor Kinosian (0002360612)             True
Getting Mary Tiller (0001068165)                          True
Getting Mary Tiller (0001200090)                          True
Getting Mary Telari (0001610535)                          True
Getting Mary Bradfield-Taylor (0001782604)                True
Getting Maria Collina (0001787387)                        True
Getting Mauro Marcolin (0001441511)                       True
Getting Trakmatik (0003303532)                            True
Getting Pud Brown Trio (0004042413)                       True
Getting Michelle Couture (0002194149)                     True
Getting Michele G Catri (0003446467)                   

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Jason Scott (0001386824)                          True
Getting Jason Scott (0001734775)                          True
Getting Jason Scott (0002176752)                          True
Getting Jason Scott (0002614958)                          True
Getting Dillion Hodges (0003438855)                       True
Getting L.C. Bell (0003221132)                            True
Getting Jean Luc Bell Journée (0002923394)                True
Getting Leigh Bell & the Chimes (0003337071)              True
Getting Kate Engler Band (0002117214)                     True
Getting Kate Gee Band (0004039771)                        True
Getting Gordon & Rogers' Inter Urban Rhythm Band (0001200495)True
Getting Cole Karmen-Green (0003502421)                    True
Getting Corron Cole (0000684177)                          True
Getting Coron Cole (0002038895)                           True
Getting Gabe Reali (0003923814)                           True
Getting GabeReal (0000494388)                       

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

Getting Musicline (0003586287)                            True
Getting MassQline (0003904382)                            True
Getting Mescalin (0004033358)                             True
Getting Mischlane Melton (0001564150)                     True
Getting Masculine Feminine (0001583832)                   True
Getting Mescaline Drive (0001586674)                      True
Getting The Mescaline Babies (0002887132)                 True
Getting Mescalin Baby (0003177948)                        True
Getting Tommy Mescaline (0001198127)                      True
Getting Maurizio Borgioni (0001658499)                    True
Getting Ilonka Rosvaenge (0001799675)                     

In [ ]:
from lib.allmusic import moveLocalFiles

In [ ]:
moveLocalFiles()